#### Reopening Index

In [1]:
# We can reopen the index without re-computing embeddings:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer("all-MiniLM-L6-v2")

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
# We still load the embedding model to encode the query, 
# but we don't re-embed all the documents. No fit call needed,
# because the index is already built and waiting on disk.

query_vector = model.encode("How do I run Kafka?")
results = vs_index.search(query_vector, num_results=5)

results

[{'id': '5ca6890c1a',
  'course': 'data-engineering-zoomcamp',
  'section': 'Module 7: Streaming',
  'question': 'Java Kafka: How to run producer/consumer/kstreams/etc in terminal',
  'answer': 'In the project directory, run:\n\n```bash\njava -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java\n```'},
 {'id': 'cd8a62fc55',
  'course': 'data-engineering-zoomcamp',
  'section': 'Module 7: Streaming',
  'question': 'Java Kafka: When running the producer/consumer/etc java scripts, no results retrieved or no message sent',
  'answer': 'For example, when running `JsonConsumer.java`, you might see:\n\n```\nConsuming form kafka started\n\nRESULTS:::0\n\nRESULTS:::0\n\nRESULTS:::0\n```\n\nOr when running `JsonProducer.java`, you might encounter:\n\n```\nException in thread "main" java.util.concurrent.ExecutionException: org.apache.kafka.common.errors.SaslAuthenticationException: Authentication failed\n```\n\n**Solution:**\n\n1. Ensure the `StreamsConfig.BO

#### Using sqlitesearch vector search in RAG

In [4]:
from rag_helper import RAGVector
from openai import OpenAI

openai_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

In [5]:
vector_assistant = RAGVector(
    embedder=model,
    index = vs_index,
    llm_client = openai_client
)

vector_assistant.rag("the program has already begun, can I still sign up?")

"According to the program information:\n\n*   If you've just discovered the course but want to join it, yes, you can still sign up for the current program (as long as submissions are still being accepted), which is indicated in the following General Course-Related Questions\n    A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.\n*   You don't need to register before joining."

In [6]:
vs_index.close()

#### Comparing minsearch and sqlitesearch for vector search
Here is how the two compare:

* minsearch VectorSearch: in-memory (numpy), exact cosine similarity, must re-compute embeddings on startup, good for experiments and notebooks
* sqlitesearch VectorSearchIndex: persistent (SQLite .db file), ANN (LSH/IVF/HNSW) with exact rerank, can open an existing index, good for projects and persistence